# W7-X Neutronics Showcase

End-to-end demonstration of basalt's role in a fusion neutronics workflow. We take the Proxima Fusion W7-X stellarator fixture (Parasolid input), produce a tetrahedral mesh annotated for DAGMC, convert it through stellarmesh, then run an OpenMC fixed-source DT-neutron calculation against four tallies.

**Prerequisites:**

- a working basalt install (requires a Simmetrix SimModSuite license and the corresponding module tarballs — see [install](../../install.rst))
- the W7-X fixture at `tests/data/scaled_w7x_stellarator.x_t` (LFS-tracked, MIT)

This notebook is committed with its outputs intact and rendered as-is by Sphinx + nbsphinx. To re-execute locally:

```bash
pixi run jupyter nbconvert --to notebook --execute --inplace \
  docs/notebooks/tutorials/w7x_neutronics.ipynb
```


In [ ]:
import logging
import tempfile
from pathlib import Path

import numpy as np
import openmc
import openmc.stats
import pyvista as pv
import matplotlib.pyplot as plt
from IPython.display import Image, display

import basalt as bslt
import stellarmesh as sm

logging.basicConfig()
for name in ("basalt", "stellarmesh"):
    logging.getLogger(name).setLevel(logging.INFO)

pv.set_jupyter_backend("static")

tmpdir = Path(tempfile.mkdtemp(prefix="w7x_neutronics_"))
print(f"Scratch directory: {tmpdir}")


## Stage 1 — Load the W7-X assembly

basalt reads the Parasolid (`.x_t`) export of the W7-X model along with its NX-attribute sidecar JSON. The sidecar carries the body centroids and `DB_PART_NAME` attributes that basalt uses to populate `material=` slugs in the output mesh — see the [format reference](../../format.rst) for the resolution rules.


In [ ]:
fixture = Path("../../../tests/data/scaled_w7x_stellarator.x_t").resolve()
assert fixture.exists(), f"Fixture not found: {fixture}"

model = bslt.Model.from_parasolid_file(str(fixture), load_nx_attrs=True)
bslt.print_hierarchy(model)


## Stage 2 — Imprint and mesh

`make_non_manifold_model` performs the boolean imprint and merge so adjacent volumes share faces — required for a conformal mesh. The mesh size is set to 10 cm with curvature-driven refinement. This is intentionally coarse for a tutorial (lower particle counts read meaningful results at this resolution).


In [ ]:
nm_model = model.make_non_manifold_model()

mesh_case = bslt.MeshCase(nm_model)
mesh_case.set_size(0.1)                              # 10 cm baseline
mesh_case.set_curvature_refinement(0.5, relative=True)
mesh_case.set_proximity_refinement(2.0)

surface_mesh = bslt.SurfaceMesh.from_model(nm_model, mesh_case)
volume_mesh = bslt.VolumeMesh.from_surface_mesh(surface_mesh)
print("Meshing complete.")


## Stage 3 — Write the Gmsh `.msh`

This `.msh` is the integration contract between basalt and stellarmesh — see the [stellarmesh format spec](https://stellarmesh.readthedocs.io/en/latest/format.html) for the canonical schema and [basalt's format reference](../../format.rst) for what basalt populates. We pass `scale_factor=100.0` to convert basalt's native metres to OpenMC's centimetres.


In [ ]:
msh_path = tmpdir / "w7x.msh"
volume_mesh.write_msh(str(msh_path), scale_factor=100.0)
print(f"Wrote {msh_path} ({msh_path.stat().st_size / 1e6:.1f} MB)")


## Stage 4 — Stellarmesh: `.msh` → DAGMC

Stellarmesh reads the `.msh` and produces a DAGMC `.h5m` file (the geometry input for OpenMC) plus an unstructured MOAB mesh that we'll use as a tally mesh. The DAGMC `mat:<slug>` groups carry the body identifiers basalt put into `material=` — we map them to physical materials in the next stage.


In [ ]:
sm_mesh = sm.Mesh(str(msh_path))
dagmc_model = sm.DAGMCModel.from_mesh(sm_mesh)
dagmc_path = tmpdir / "dagmc.h5m"
dagmc_model.write(str(dagmc_path))

tally_mesh_path = tmpdir / "w7x_tally_mesh.h5m"
sm.MOABVolumeModel.from_mesh(sm_mesh).write(str(tally_mesh_path))

print(f"DAGMC: {dagmc_path}")
print(f"Tally mesh: {tally_mesh_path}")


## Stage 5 — Material assignment

The `material=<slug>` in basalt's output identifies a **body**, not a physical material — see [format § The `material` slug identifies a body, not a material](../../format.rst). Here we map each body category in the W7-X fixture (`COIL_*`, `FIRSTWALL`, `BLANKET`, `VESSEL`, `PLASMA`, `GAP`) to a vanilla `openmc.Material`.

Materials are illustrative, not engineering-realistic. The blanket uses **natural-lithium FLiBe**, which gives a tritium breeding ratio < 1 (Li-7 dominates and has very low (n,t) cross-section); a real breeding blanket would use ~30–60% Li-6 enrichment. See the closing scalar table for the headline value.


In [ ]:
import re

def make_copper():
    m = openmc.Material(name="copper")
    m.add_element("Cu", 1.0)
    m.set_density("g/cm3", 8.96)
    return m

def make_tungsten():
    m = openmc.Material(name="tungsten")
    m.add_element("W", 1.0)
    m.set_density("g/cm3", 19.3)
    return m

def make_flibe():
    m = openmc.Material(name="flibe")
    m.add_element("Li", 2.0)
    m.add_element("Be", 1.0)
    m.add_element("F", 4.0)
    m.set_density("g/cm3", 1.94)
    return m

def make_ss316l():
    m = openmc.Material(name="ss316l")
    m.add_element("Fe", 0.6815, "wo")
    m.add_element("Cr", 0.17,   "wo")
    m.add_element("Ni", 0.12,   "wo")
    m.add_element("Mo", 0.025,  "wo")
    m.add_element("Mn", 0.02,   "wo")
    m.add_element("Si", 0.0035, "wo")
    m.set_density("g/cm3", 7.99)
    return m

# Pattern-match body_name → material factory.
BODY_NAME_TO_MATERIAL = {
    re.compile(r"^COIL_\d+$"):    make_copper,
    re.compile(r"^FIRSTWALL$"):    make_tungsten,
    re.compile(r"^BLANKET$"):      make_flibe,
    re.compile(r"^VESSEL$"):       make_ss316l,
    re.compile(r"^PLASMA$"):       None,       # void
    re.compile(r"^GAP$"):          None,       # void
}

SLUG_PATTERN = re.compile(r"^[A-Za-z0-9_]+\.b\d+$")

def material_for_slug(slug, body_name):
    for pat, factory in BODY_NAME_TO_MATERIAL.items():
        if pat.match(body_name):
            if factory is None:
                return None
            m = factory()
            m.name = slug
            return m
    raise KeyError(
        f"body_name {body_name!r} (slug {slug!r}) has no entry in "
        f"BODY_NAME_TO_MATERIAL"
    )

def check_material_coverage(dagmc_model, sidecar):
    mat_groups = [g.name for g in dagmc_model.groups if g.name.startswith("mat:")]
    slugs = [n.removeprefix("mat:") for n in mat_groups]
    assert slugs, "No mat:* groups in DAGMC model"

    none_slugs = [s for s in slugs if s == "None"]
    if none_slugs:
        raise RuntimeError(
            f"{len(none_slugs)} region(s) have slug 'None' — ambiguous part "
            "ownership (typically shared faces after non-manifold merge)."
        )

    for slug in slugs:
        if not SLUG_PATTERN.match(slug):
            raise ValueError(f"Slug {slug!r} doesn't match expected format")
        if len(slug) > 28:
            raise ValueError(
                f"Material name exceeds MOAB 28-char limit: {slug!r} "
                f"({len(slug)} chars)"
            )
        if slug not in sidecar:
            raise KeyError(f"DAGMC slug {slug!r} not in sidecar")
    return slugs

sidecar_path = Path("../../../tests/data/scaled_w7x_stellarator_attrs.json").resolve()
sidecar_data = bslt.load_material_metadata(str(sidecar_path))

slugs = check_material_coverage(dagmc_model, sidecar_data)
print(f"Validated {len(slugs)} material slugs.")

materials_list = []
for slug in slugs:
    body_name = sidecar_data[slug].get("body_name") or sidecar_data[slug].get("db_part_name", "")
    mat = material_for_slug(slug, body_name)
    if mat is not None:
        materials_list.append(mat)

materials = openmc.Materials(materials_list)
print(f"Built {len(materials_list)} openmc.Material objects "
      f"({len(slugs) - len(materials_list)} void regions).")


## Stage 6 — Plasma source

A 14.1 MeV monoenergetic, isotropic DT-neutron source uniformly distributed over the PLASMA bounding box. Particles that start outside the PLASMA cell stream through vacuum harmlessly; the physics is unchanged. For a real plasma source (Maxwellian energy, flux-surface-weighted spatial distribution), basalt is the wrong abstraction layer — those helpers live in plasma-physics packages.


In [ ]:
dagmc_universe = openmc.DAGMCUniverse(dagmc_path)
geometry = openmc.Geometry(dagmc_universe)

plasma_slugs = [s for s in slugs if s.startswith("PLASMA")]
if not plasma_slugs:
    raise RuntimeError("No PLASMA region in DAGMC — check sidecar/fixture.")

PLASMA_BBOX_PAD_M = 4.0
plasma_centroids = np.array([sidecar_data[s]["centroid"] for s in plasma_slugs])
bbox_min = (plasma_centroids.min(axis=0) - PLASMA_BBOX_PAD_M) * 100.0
bbox_max = (plasma_centroids.max(axis=0) + PLASMA_BBOX_PAD_M) * 100.0

source = openmc.IndependentSource(
    space=openmc.stats.Box(bbox_min, bbox_max),
    energy=openmc.stats.Discrete([14.1e6], [1.0]),
    angle=openmc.stats.Isotropic(),
)

print(f"Source bbox (cm): {bbox_min} → {bbox_max}")


## Stage 7 — Tallies

Four tallies make up the showcase:

1. **Unstructured-mesh neutron flux** on the MOAB volume tally mesh (E > 0.1 MeV) — the headline visualization.
2. **Tritium breeding ratio (TBR)** on the BLANKET region (Li-6 (n,Xt) reaction rate / source rate).
3. **Coil heating** summed over all `COIL_*` cells.
4. **First-wall fast flux** integrated over FIRSTWALL.


In [ ]:
tallies = openmc.Tallies()

# (1) Unstructured-mesh flux
umesh = openmc.UnstructuredMesh(str(tally_mesh_path), library="moab")
fast_filter = openmc.EnergyFilter([0.1e6, 20.0e6])
mesh_filter = openmc.MeshFilter(umesh)
t_flux_mesh = openmc.Tally(name="flux_mesh")
t_flux_mesh.filters = [mesh_filter, fast_filter]
t_flux_mesh.scores = ["flux"]
tallies.append(t_flux_mesh)

# (2) TBR — over BLANKET cells, Li-6 (n,Xt)
blanket_slugs = [s for s in slugs if s.startswith("BLANKET")]
blanket_cells = [c for c in dagmc_universe.cells.values()
                 if getattr(c.fill, "name", None) in blanket_slugs]
if blanket_cells:
    t_tbr = openmc.Tally(name="tbr")
    t_tbr.filters = [openmc.CellFilter(blanket_cells)]
    t_tbr.nuclides = ["Li6"]
    t_tbr.scores = ["(n,Xt)"]
    tallies.append(t_tbr)

# (3) Coil heating
coil_slugs = [s for s in slugs if s.startswith("COIL_")]
coil_cells = [c for c in dagmc_universe.cells.values()
              if getattr(c.fill, "name", None) in coil_slugs]
if coil_cells:
    t_heating = openmc.Tally(name="coil_heating")
    t_heating.filters = [openmc.CellFilter(coil_cells)]
    t_heating.scores = ["heating"]
    tallies.append(t_heating)

# (4) First-wall fast flux
fw_slugs = [s for s in slugs if s.startswith("FIRSTWALL")]
fw_cells = [c for c in dagmc_universe.cells.values()
            if getattr(c.fill, "name", None) in fw_slugs]
if fw_cells:
    t_fw = openmc.Tally(name="firstwall_flux")
    t_fw.filters = [openmc.CellFilter(fw_cells), fast_filter]
    t_fw.scores = ["flux"]
    tallies.append(t_fw)

print(f"Built {len(tallies)} tallies.")


## Stage 8 — Run

1 million particles, 10 batches, fixed-source mode, RNG seed pinned for reproducibility. On a modern laptop this runs in 10–30 minutes; the committed notebook outputs are the result of one such run.


In [ ]:
settings = openmc.Settings()
settings.run_mode = "fixed source"
settings.particles = 1_000_000
settings.batches = 10
settings.source = source
settings.seed = 1

geometry.export_to_xml(str(tmpdir / "geometry.xml"))
materials.export_to_xml(str(tmpdir / "materials.xml"))
settings.export_to_xml(str(tmpdir / "settings.xml"))
tallies.export_to_xml(str(tmpdir / "tallies.xml"))

statepoint_path = openmc.run(cwd=str(tmpdir))
print(f"Run complete. Statepoint: {statepoint_path}")


## Stage 9 — Results

The headline figure is a 3D volume render of the fast-flux tally mapped onto the W7-X mesh. Below it: a scalar table with TBR, total coil heating, and integrated first-wall flux.


In [ ]:
with openmc.StatePoint(statepoint_path) as sp:
    flux_mesh_tally = sp.get_tally(name="flux_mesh")
    flux_values = flux_mesh_tally.mean.flatten()

    pv_mesh = pv.read(str(tally_mesh_path))
    pv_mesh.cell_data["flux"] = flux_values[: pv_mesh.n_cells]

    pl = pv.Plotter(off_screen=True, window_size=(1024, 768))
    pl.add_volume(pv_mesh, scalars="flux", opacity="sigmoid", cmap="viridis")
    pl.camera_position = "iso"
    pl.add_axes()
    pl.add_scalar_bar(title="Fast flux (n/cm²/source)")
    screenshot_path = tmpdir / "flux_3d.png"
    pl.screenshot(str(screenshot_path))
    pl.close()
    display(Image(str(screenshot_path), width=900))

    def scalar(name):
        try:
            return float(sp.get_tally(name=name).mean.sum())
        except (LookupError, KeyError):
            return None

    tbr = scalar("tbr")
    coil_heating = scalar("coil_heating")
    fw_flux = scalar("firstwall_flux")

print("--- Scalar results (per source neutron) ---")
print(f"  TBR (natural Li FLiBe):  {tbr:.4f}" if tbr is not None else "  TBR:  n/a")
print(f"  Total coil heating:      {coil_heating:.4e} eV" if coil_heating is not None else "  Coil heating: n/a")
print(f"  First-wall fast flux:    {fw_flux:.4e} n/cm²" if fw_flux is not None else "  First-wall flux: n/a")
print()
print("Note: TBR < 1 is expected with natural lithium. A real breeding")
print("blanket would use ~30–60% Li-6 enrichment, pushing TBR > 1.")


## Closing

This notebook ran the complete basalt → stellarmesh → OpenMC pipeline against a public W7-X fixture, producing a 3D fast-flux visualization and three engineering-relevant scalars in one literate file. Things to try next:

- Reduce the mesh size (`mesh_case.set_size(0.05)`) for sharper geometry resolution — runtime scales steeply.
- Swap the blanket material to enriched-Li FLiBe and watch TBR cross 1.
- Replace the source with a more physically realistic plasma profile.
- Reuse the meshing pipeline against your own CAD — load any Parasolid assembly with `Model.from_parasolid_file`.

See [basalt usage](../../usage.rst), [format reference](../../format.rst), and [stellarmesh tutorials](https://stellarmesh.readthedocs.io/en/latest/tutorials.html) for next steps.
